# Lab 08-12 · Reading an FPGA Flow's Reports

Companion to chapter [08-12](../../chapters/08-12-fpga-implementation-flow.md). Handout: [README.md](README.md).

The reports in `reports/` were produced by `run_flow.py` (Yosys 0.71 and nextpnr-ice40 0.11.1, iCE40 UP5K in the SG48 package, seed 1). This notebook only reads them, so it runs without the tools; to regenerate them, install `yowasp-yosys` and `yowasp-nextpnr-ice40` and run the script. Cells marked **🔮 Predict** ask for a prediction *before* running.

In [1]:
import json
from pathlib import Path
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

REPORTS = Path("reports")
NAMES = {"add_plus": "adder, a + b", "add_rca": "adder, gates", "mul_dsp": "multiplier, DSP",
         "mul_lut": "multiplier, LUTs", "fsm_auto": "FSM, auto (one-hot)", "fsm_bin": "FSM, binary"}


def load(name):
    r = json.loads((REPORTS / f"{name}.json").read_text())
    fmax = next(iter(r["fmax"].values()))["achieved"]
    used = {k: v["used"] for k, v in r["utilization"].items() if v["used"]}
    path = r["critical_paths"][0]["path"]
    parts = {}
    for seg in path:
        parts[seg["type"]] = parts.get(seg["type"], 0.0) + seg["delay"]
    hops = {t: sum(1 for seg in path if seg["type"] == t) for t in ("logic", "routing")}
    return dict(fmax=fmax, used=used, parts=parts, hops=hops)


R = {n: load(n) for n in NAMES}

## Part 1 · Utilization and maximum frequency

**🔮 Predict (the chapter's guided exercise):** how much faster is the `a + b` adder than the same adder written as gates, and what fraction of the gate-level adder's critical path is routing?

In [2]:
print(f"{'design':22s} {'cells':>6s} {'DSP':>4s} {'f_max (MHz)':>12s} {'period (ns)':>12s}")
for n, label in NAMES.items():
    r = R[n]
    print(f"{label:22s} {r['used']['ICESTORM_LC']:6d} {r['used'].get('ICESTORM_DSP', 0):4d} {r['fmax']:12.2f} {1000 / r['fmax']:12.2f}")
speed = R["add_plus"]["fmax"] / R["add_rca"]["fmax"]
p = R["add_rca"]["parts"]
print(f"\ncarry chain over gates: {speed:.2f}x;  routing share of the gate-level adder's path: "
      f"{p['routing'] / sum(p.values()):.1%}")

design                  cells  DSP  f_max (MHz)  period (ns)
adder, a + b              175    0        52.09        19.20
adder, gates              214    0        17.44        57.34
multiplier, DSP            79    1       100.88         9.91
multiplier, LUTs          765    0        26.10        38.32
FSM, auto (one-hot)        29    0        79.23        12.62
FSM, binary                37    0        54.20        18.45

carry chain over gates: 2.99x;  routing share of the gate-level adder's path: 64.5%


About three times faster, with routing about two thirds of the slow path. The gate-level adder's carry passes through 20 LUTs and 21 routing hops; the `+` version uses the dedicated carry chain, whose wires between neighboring cells are short and fixed.

## Part 2 · Where the time goes on each critical path

In [3]:
print(f"{'design':22s} {'clk-to-Q':>9s} {'logic':>7s} {'routing':>8s} {'setup':>6s} {'total':>7s} {'LUTs':>5s} {'hops':>5s}")
for n, label in NAMES.items():
    pr, h = R[n]["parts"], R[n]["hops"]
    total = sum(pr.values())
    assert abs(total - 1000 / R[n]["fmax"]) < 0.01, "the path total must match the reported frequency"
    print(f"{label:22s} {pr.get('clk-to-q', 0):9.2f} {pr.get('logic', 0):7.2f} {pr.get('routing', 0):8.2f} "
          f"{pr.get('setup', 0):6.2f} {total:7.2f} {h['logic']:5d} {h['routing']:5d}")

design                  clk-to-Q   logic  routing  setup   total  LUTs  hops
adder, a + b                1.39   10.17     6.41   1.23   19.20    33    34
adder, gates                1.39   18.14    36.98   0.82   57.34    20    21
multiplier, DSP             0.10    2.16     6.47   1.18    9.91     2     3
multiplier, LUTs            1.39   14.35    21.75   0.82   38.32    27    28
FSM, auto (one-hot)         1.39    2.95     7.04   1.23   12.62     3     4
FSM, binary                 1.39    5.77    11.19   0.10   18.45     5     6


In [4]:
fig, ax = plt.subplots(figsize=(7.5, 3.6))
left = [0.0] * len(NAMES)
for key in ("clk-to-q", "logic", "routing", "setup"):
    vals = [R[n]["parts"].get(key, 0.0) for n in NAMES]
    ax.barh(list(NAMES.values()), vals, left=left, label=key)
    left = [a + b for a, b in zip(left, vals)]
ax.invert_yaxis()
ax.set_xlabel("critical-path delay (ns)")
ax.legend(ncol=4, fontsize=8, loc="upper center", bbox_to_anchor=(0.5, -0.2))
plt.show()

Routing is the largest part of every critical path here except the carry-chain adder's, where the dedicated chain is counted as logic.

## Part 3 · Slack against the clock constraint

The reports were produced with a 12 MHz constraint, which every design meets. What happens at faster clocks?

In [5]:
for f_req in (12, 24, 48, 96):
    period = 1000 / f_req
    row = []
    for n in NAMES:
        slack = period - 1000 / R[n]["fmax"]
        row.append(f"{slack:+7.2f}")
    print(f"{f_req:3d} MHz:", "  ".join(row))
print("columns:", ", ".join(NAMES.values()))
passing = [NAMES[n] for n in NAMES if R[n]["fmax"] >= 48]
print("meet 48 MHz:", passing)

 12 MHz:  +64.13   +26.00   +73.42   +45.02   +70.71   +64.88
 24 MHz:  +22.47   -15.67   +31.75    +3.35   +29.05   +23.22
 48 MHz:   +1.63   -36.50   +10.92   -17.48    +8.21    +2.38
 96 MHz:   -8.78   -46.92    +0.50   -27.90    -2.20    -8.03
columns: adder, a + b, adder, gates, multiplier, DSP, multiplier, LUTs, FSM, auto (one-hot), FSM, binary
meet 48 MHz: ['adder, a + b', 'multiplier, DSP', 'FSM, auto (one-hot)', 'FSM, binary']


At 48 MHz, the frequency of the UP5K's internal oscillator, the two designs that avoided the hard resources fail; at 96 MHz only the DSP multiplier passes. A "PASS" in a report is always relative to the constraint it was given.

## Part 4 · What synthesis decided, and how much the seed matters

In [6]:
for n in ("fsm_auto", "fsm_bin"):
    lines = (REPORTS / f"{n}_synth.txt").read_text().splitlines()
    print(n, "->", " / ".join(l.strip() for l in lines if "encoding" in l))
sweep = json.loads((REPORTS / "seed_sweep.json").read_text())["fmax_mhz"]
vals = list(sweep.values())
print(f"\nseed sweep of the gate-level adder: best {max(vals):.2f}, worst {min(vals):.2f}, "
      f"mean {sum(vals) / len(vals):.2f} MHz; best over worst {max(vals) / min(vals) - 1:.2%}")

fsm_auto -> Recoding FSM `$fsm$\state$500' from module `\traffic' using `auto' encoding: / mapping auto encoding to `one-hot` for this FSM.
fsm_bin -> Recoding FSM `$fsm$\state$500' from module `\traffic' using `binary' encoding: / existing encoding is already a packed binary encoding.

seed sweep of the gate-level adder: best 17.44, worst 16.94, mean 17.34 MHz; best over worst 2.95%


Yosys chose one-hot automatically for the six-state controller and kept the forced binary encoding when asked; the binary version came out larger and slower (chapter Example 08-12.6). Changing only the placement seed moves the gate-level adder by about 3%: worth trying when a design narrowly misses timing, but no substitute for the factor of three that the carry chain gives.

**Verilog.** Run `flow_tb.v` (`python tools/run_hdl.py` from the repository root) to check that the cores measured here are functionally correct: a fast wrong adder is worth nothing.